## 🎯 Learning Objectives
* Demonstrate mastery of LangGraph's advanced features, including subgraphs, nested composition, and supervisor nodes.
* Apply time-travel debugging techniques to diagnose and resolve complex agentic system issues.
* Design and implement sophisticated multi-agent architectures, such as swarm topologies, using LangGraph.
* Integrate external tools and APIs effectively within a LangGraph-based agent system.
* Evaluate and optimize the performance and robustness of advanced AI agents.


## Final Assessment: Advanced AI Agents with LangGraph

Welcome to the final assessment for ADV-01: Building Advanced AI Agents with LangGraph. This course has equipped you with the cutting-edge skills required to design, implement, and debug complex multi-agent architectures using LangGraph, a powerful framework for orchestrating stateful, multi-actor applications.

As senior AI engineers, you've delved into sophisticated concepts such as subgraphs for modularity, nested composition for hierarchical control, supervisor nodes for dynamic orchestration, time-travel debugging for unparalleled introspection, and swarm topologies for emergent intelligence. The AI landscape of 2026 demands not just functional agents, but robust, scalable, and observable systems capable of tackling real-world challenges.

This assessment is designed to evaluate your comprehensive understanding and practical application of these advanced LangGraph features. It comprises two main parts:

1.  **Review Questions:** A set of conceptual and theoretical questions to test your grasp of the underlying principles and design patterns.
2.  **Capstone Project:** A hands-on coding challenge requiring you to build a sophisticated multi-agent system that integrates several advanced LangGraph features, simulating a real-world problem.

Your ability to articulate design choices, implement robust solutions, and demonstrate an understanding of debugging and optimization will be key to successfully completing this assessment. Good luck!


## Review Questions

Answer the following questions concisely, demonstrating your understanding of advanced LangGraph concepts and their practical implications.

1.  **Nested Subgraphs & Modularity:** Explain the primary benefits of using nested LangGraph subgraphs for modularity and reusability in large-scale agent systems. Provide a concrete example where this pattern significantly simplifies development or maintenance.

2.  **Supervisor Node vs. Router:** Describe a scenario where a 'supervisor node' is indispensable in a multi-agent system, detailing its role and how its capabilities extend beyond that of a simple conditional router. What specific state management features does a supervisor often leverage?

3.  **Time-Travel Debugging:** How does LangGraph's inherent state management facilitate 'time-travel debugging'? Provide an example of a complex, intermittent bug in a multi-agent system that would be exceptionally difficult to diagnose without this feature, and explain how time-travel debugging would aid in its resolution.

4.  **Swarm Topologies:** Compare and contrast a 'sequential' multi-agent workflow with a 'swarm topology' in LangGraph. When would you choose one over the other, considering factors like problem complexity, agent autonomy, and desired emergent behavior?

5.  **Asynchronous Operations & State:** Discuss the implications of integrating asynchronous operations (e.g., non-blocking API calls, parallel computations) within LangGraph agents, particularly concerning state updates, concurrent execution, and potential race conditions. How does LangGraph help manage these complexities?

6.  **2026 Tool Integration:** In the context of 2026 AI tools, how would you integrate a specialized, high-performance vector database (e.g., Qdrant, Milvus, Pinecone) into a LangGraph agent for advanced Retrieval-Augmented Generation (RAG)? Outline the key steps and considerations for efficient data flow and query optimization.

7.  **Production Deployment:** What are the key considerations for deploying a complex LangGraph application to production, focusing on scalability, observability (logging, tracing), and fault tolerance? Mention specific LangGraph features or external tools that would be crucial for each aspect.


## Capstone Project: Automated Research & Development Assistant for Novel Material Discovery

**Scenario:** You are tasked with building an advanced AI R&D assistant for a leading materials science company. The goal is to accelerate the discovery of novel materials with specific properties. The assistant should be able to autonomously research existing knowledge, hypothesize new material compositions, evaluate their potential using a simulated predictor, and iteratively refine its proposals until a satisfactory material is found or all avenues are exhausted.

**Objective:** Design and implement a multi-agent system using LangGraph that can:

1.  **Understand Initial Request:** Take a high-level material property requirement (e.g., 'a biodegradable polymer with high tensile strength', 'a superconducting alloy at room temperature') as input.
2.  **Research Phase:** An agent or subgraph should perform a 'literature review' to gather existing knowledge, relevant chemical principles, and known materials with similar properties. This phase should identify key elements, structures, or synthesis methods.
3.  **Hypothesis Generation Phase:** Based on the research, another agent or subgraph should propose novel material compositions or modifications. These proposals should be structured (e.g., a list of elements and their ratios, a molecular structure description).
4.  **Simulation & Evaluation Phase:** A dedicated agent will use a 'Material Properties Predictor' tool (simulated by a Python function) to evaluate the proposed material against the desired criteria. The tool will return a score or a set of properties.
5.  **Refinement & Iteration:** If the proposed material does not meet the criteria, the system should intelligently refine its hypothesis based on the simulation feedback and iterate through the generation and evaluation steps. This might involve adjusting compositions, trying different structural motifs, or exploring alternative synthesis routes.
6.  **Reporting:** Once a satisfactory material is found, or after a predefined number of iterations, the system should generate a comprehensive report summarizing the initial request, the research findings, all proposed materials (successful and failed), the evaluation results, and the final recommended material (if any).

**Key Requirements & LangGraph Features to Utilize:**

*   **Supervisor Node:** To orchestrate the overall workflow, deciding when to transition between research, hypothesis generation, evaluation, and refinement.
*   **Nested Subgraphs:** Implement the 'Research Phase' and 'Hypothesis Generation Phase' as distinct subgraphs to promote modularity and reusability.
*   **Tool Calling:** Integrate the `material_predictor_tool` as an external function call within your agents.
*   **Conditional Routing:** Implement dynamic routing based on the output of the `material_predictor_tool` (e.g., if criteria met, go to reporting; else, go to refinement).
*   **Shared State:** Effectively manage and update a shared graph state to accumulate research findings, material proposals, and evaluation results across different agents and iterations.
*   **Iterative Process:** Design the graph to support multiple cycles of hypothesis generation and evaluation.

**Deliverables:**

*   A complete, runnable LangGraph definition for the R&D Assistant.
*   Clear agent prompts and tool definitions.
*   An example invocation demonstrating the assistant's capabilities for a given material property requirement.
*   Comments explaining the design choices and LangGraph features used.


In [ ]:
import os
from typing import List, Dict, Any, TypedDict, Optional

# Ensure you have the necessary environment variables set for your LLM provider
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_API_KEY"] = "YOUR_LANGCHAIN_API_KEY"

from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolExecutor, ToolNode

# --- 1. Define Graph State ---
# This defines the state of our graph. It is typed for clarity.
class ResearchState(TypedDict):
    initial_request: str
    research_findings: List[str]
    material_proposals: List[Dict[str, Any]]
    current_proposal: Optional[Dict[str, Any]]
    evaluation_results: List[Dict[str, Any]]
    iterations: int
    max_iterations: int
    messages: List[BaseMessage]

# --- 2. Define Tools ---
@tool
def material_properties_predictor(material_composition: Dict[str, Any]) -> Dict[str, Any]:
    """
    Simulates the prediction of material properties based on its composition.
    Input should be a dictionary with 'elements' (list of str) and 'ratios' (list of float).
    Returns a dictionary with predicted properties and a 'score' indicating suitability.
    """
    print(f"\n--- Simulating material: {material_composition} ---")
    elements = material_composition.get("elements", [])
    ratios = material_composition.get("ratios", [])

    # Simple mock logic for demonstration
    score = 0.0
    properties = {"tensile_strength": 0, "biodegradability": 0, "conductivity": 0}

    if "polymer" in material_composition.get("type", "").lower():
        properties["biodegradability"] = 0.8 # Assume good biodegradability for polymers
        if "carbon" in elements and "hydrogen" in elements:
            properties["tensile_strength"] = sum(ratios) * 10 # Simple heuristic
            score += properties["tensile_strength"] * 0.1
    elif "alloy" in material_composition.get("type", "").lower():
        properties["conductivity"] = sum(ratios) * 50 # Simple heuristic
        if "copper" in elements and "aluminum" in elements:
            score += properties["conductivity"] * 0.05

    # Add some randomness to make it less deterministic
    import random
    score += random.uniform(-5, 5)
    score = max(0, min(100, score)) # Cap score between 0 and 100

    print(f"Predicted properties: {properties}, Score: {score:.2f}")
    return {"predicted_properties": properties, "score": score, "composition": material_composition}

# List of all tools available to agents
tools = [material_properties_predictor]
tool_executor = ToolExecutor(tools)

# --- 3. Initialize LLM ---
# Using a powerful LLM for agent reasoning
llm = ChatOpenAI(model="gpt-4o-2024-05-13", temperature=0.5)

# --- 4. Define Agent Prompts ---
# Base prompt for agents that can use tools
agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. You have access to the following tools: {tools}"),
    MessagesPlaceholder(variable_name="messages"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# --- 5. Define Agent Nodes (Skeletons) ---
# You will need to define the actual agent logic and integrate them into the graph.
# Consider using `create_react_agent` or similar for tool-using agents.

def research_agent_node(state: ResearchState) -> ResearchState:
    """Agent responsible for researching existing materials and principles."""
    print("\n--- Research Agent Activated ---")
    # TODO: Implement research logic. This agent should use LLM to process initial_request
    # and potentially simulate web searches (not implemented here for brevity) to populate
    # research_findings. It should update the 'research_findings' in the state.
    # Example: LLM call to summarize initial request and suggest research areas.
    # For this template, we'll just add a placeholder finding.
    current_findings = state.get("research_findings", [])
    current_findings.append(f"Initial research for '{state['initial_request']}' suggests exploring polymer chemistry and carbon-based structures.")
    return {"research_findings": current_findings, "messages": state["messages"] + [HumanMessage(content="Research completed.")]}

def hypothesis_agent_node(state: ResearchState) -> ResearchState:
    """
    Agent responsible for generating new material hypotheses based on research findings.
    It should output a structured material_composition dictionary.
    """
    print("\n--- Hypothesis Agent Activated ---")
    # TODO: Implement hypothesis generation logic. This agent should use LLM to synthesize
    # research_findings and generate a 'current_proposal' in the format expected by the tool.
    # It should also add the proposal to 'material_proposals' list.
    # For this template, we'll generate a simple placeholder proposal.
    new_proposal = {
        "type": "polymer",
        "elements": ["carbon", "hydrogen", "oxygen"],
        "ratios": [0.6, 0.3, 0.1],
        "notes": "A basic polymer structure based on initial research."
    }
    proposals = state.get("material_proposals", [])
    proposals.append(new_proposal)
    return {"current_proposal": new_proposal, "material_proposals": proposals, "messages": state["messages"] + [HumanMessage(content="Hypothesis generated.")]}

def evaluate_agent_node(state: ResearchState) -> ResearchState:
    """Agent responsible for evaluating the current material proposal using the predictor tool."""
    print("\n--- Evaluation Agent Activated ---")
    # TODO: This agent should call the `material_properties_predictor` tool
    # with `state['current_proposal']` and update `evaluation_results`.
    # For this template, we'll simulate a tool call.
    if state.get("current_proposal"):
        tool_output = material_properties_predictor.invoke(state["current_proposal"])
        eval_results = state.get("evaluation_results", [])
        eval_results.append(tool_output)
        return {"evaluation_results": eval_results, "messages": state["messages"] + [HumanMessage(content="Material evaluated.")]}
    return state # No proposal to evaluate

def supervisor_node(state: ResearchState) -> str:
    """
    Supervisor node to decide the next step based on the current state.
    Routes between research, hypothesis, evaluate, refine, or end.
    """
    print("\n--- Supervisor Activated ---")
    # TODO: Implement the routing logic based on the state.
    # This is where you'll check iteration count, evaluation scores, etc.
    # For this template, we'll use a simplified logic.
    if state["iterations"] >= state["max_iterations"]:
        print("Max iterations reached. Ending.")
        return "end"

    if not state.get("research_findings"):
        print("No research findings yet. Routing to research.")
        return "research"

    last_eval = state["evaluation_results"][-1] if state["evaluation_results"] else None
    if last_eval and last_eval.get("score", 0) >= 70: # Example success threshold
        print(f"Material score {last_eval['score']:.2f} meets criteria. Ending.")
        return "report"
    elif state.get("current_proposal") and last_eval and last_eval.get("score", 0) < 70:
        print(f"Material score {last_eval['score']:.2f} too low. Routing to refine.")
        return "refine"
    else:
        print("Ready for new hypothesis. Routing to hypothesis.")
        return "hypothesis"

def refine_agent_node(state: ResearchState) -> ResearchState:
    """
    Agent responsible for refining the material hypothesis based on previous evaluation feedback.
    """
    print("\n--- Refinement Agent Activated ---")
    # TODO: Implement refinement logic. This agent should take the last evaluation result
    # and the current proposal, and generate a *new* `current_proposal` that attempts
    # to improve upon the previous one. Update `material_proposals` list.
    # For this template, we'll just slightly modify the last proposal.
    last_proposal = state["current_proposal"]
    if last_proposal:
        refined_proposal = last_proposal.copy()
        # Simple refinement: adjust ratios slightly
        import random
        if refined_proposal.get("ratios"):
            refined_proposal["ratios"] = [max(0.05, r + random.uniform(-0.05, 0.05)) for r in refined_proposal["ratios"]]
            total_ratio = sum(refined_proposal["ratios"])
            refined_proposal["ratios"] = [r / total_ratio for r in refined_proposal["ratios"]]
        refined_proposal["notes"] = f"Refined from previous attempt after score {state['evaluation_results'][-1]['score']:.2f}."

        proposals = state.get("material_proposals", [])
        proposals.append(refined_proposal)
        return {"current_proposal": refined_proposal, "material_proposals": proposals, "messages": state["messages"] + [HumanMessage(content="Hypothesis refined.")]}
    return state

def report_agent_node(state: ResearchState) -> ResearchState:
    """
    Agent responsible for generating the final summary report.
    """
    print("\n--- Reporting Agent Activated ---")
    # TODO: Implement report generation logic. This agent should summarize all findings,
    # proposals, and evaluation results into a comprehensive report.
    final_report = f"""
    Material R&D Assistant Report
    --------------------------------
    Initial Request: {state['initial_request']}

    Research Findings:
    {state['research_findings']}

    All Proposals & Evaluations:
    """
    for i, proposal in enumerate(state['material_proposals']):
        report_entry = f"    Proposal {i+1}: {proposal}"
        if i < len(state['evaluation_results']):
            eval_res = state['evaluation_results'][i]
            report_entry += f" -> Score: {eval_res['score']:.2f}, Properties: {eval_res['predicted_properties']}"
        final_report += report_entry + "\n"

    successful_material = next((res for res in state['evaluation_results'] if res.get('score', 0) >= 70), None)
    if successful_material:
        final_report += f"\nRecommended Material: {successful_material['composition']} (Score: {successful_material['score']:.2f})\n"
    else:
        final_report += "\nNo material met the criteria within the given iterations.\n"

    print(final_report)
    return {"messages": state["messages"] + [HumanMessage(content=final_report)]}

# --- 6. Build the Graph (Skeleton) ---
# You will need to define the graph structure here, including nodes, edges, and entry/exit points.
# Use the defined agent nodes and the supervisor node.

# Define the graph
workflow = StateGraph(ResearchState)

# Add nodes for each agent/step
# workflow.add_node("research", research_agent_node)
# workflow.add_node("hypothesis", hypothesis_agent_node)
# workflow.add_node("evaluate", evaluate_agent_node)
# workflow.add_node("refine", refine_agent_node)
# workflow.add_node("report", report_agent_node)
# workflow.add_node("supervisor", supervisor_node)

# Set the entry point
# workflow.set_entry_point("supervisor")

# Add conditional edges from the supervisor
# workflow.add_conditional_edges(
#     "supervisor",
#     lambda state: state["next_step"], # This needs to be determined by the supervisor_node logic
#     {
#         "research": "research",
#         "hypothesis": "hypothesis",
#         "evaluate": "evaluate",
#         "refine": "refine",
#         "report": "report",
#         "end": END
#     }
# )

# Add edges from agents back to the supervisor for decision making
# workflow.add_edge("research", "supervisor")
# workflow.add_edge("hypothesis", "supervisor")
# workflow.add_edge("evaluate", "supervisor")
# workflow.add_edge("refine", "supervisor")
# workflow.add_edge("report", END) # Report is a terminal node

# Compile the graph
# app = workflow.compile()

# --- 7. Example Invocation (Placeholder) ---
# initial_state = {
#     "initial_request": "a biodegradable polymer with high tensile strength",
#     "research_findings": [],
#     "material_proposals": [],
#     "current_proposal": None,
#     "evaluation_results": [],
#     "iterations": 0,
#     "max_iterations": 5,
#     "messages": [HumanMessage(content="Starting R&D for new material.")]
# }

# for s in app.stream(initial_state):
#     print(s)

# print("\nFinal State:")
# print(app.get_state(initial_state).values)


In [ ]:
import os
from typing import List, Dict, Any, TypedDict, Optional

# Ensure you have the necessary environment variables set for your LLM provider
# For LangChain tracing and monitoring (highly recommended for advanced agents)
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_API_KEY"] = "YOUR_LANGCHAIN_API_KEY"
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY" # Or other LLM provider keys

from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolExecutor, ToolNode
from langchain.agents import create_tool_calling_agent

# --- 1. Define Graph State ---
# This defines the state of our graph. It is typed for clarity and robustness.
class ResearchState(TypedDict):
    initial_request: str
    research_findings: List[str]
    material_proposals: List[Dict[str, Any]]
    current_proposal: Optional[Dict[str, Any]]
    evaluation_results: List[Dict[str, Any]]
    iterations: int
    max_iterations: int
    messages: List[BaseMessage]
    # Add a field for the supervisor to explicitly set the next node
    next_node: str

# --- 2. Define Tools ---
@tool
def material_properties_predictor(material_composition: Dict[str, Any]) -> Dict[str, Any]:
    """
    Simulates the prediction of material properties based on its composition.
    Input should be a dictionary with 'type' (str, e.g., 'polymer', 'alloy'),
    'elements' (list of str), 'ratios' (list of float), and 'notes' (str).
    Returns a dictionary with predicted properties and a 'score' indicating suitability (0-100).
    """
    print(f"\n--- Simulating material: {material_composition} ---")
    material_type = material_composition.get("type", "").lower()
    elements = material_composition.get("elements", [])
    ratios = material_composition.get("ratios", [])

    # Simple mock logic for demonstration, simulating complex material science
    score = 0.0
    properties = {"tensile_strength": 0.0, "biodegradability": 0.0, "conductivity": 0.0, "stability": 0.0}

    # Heuristics based on common material types and elements
    if "polymer" in material_type:
        properties["biodegradability"] = 0.7 + (0.3 * (elements.count("carbon") + elements.count("oxygen")) / max(1, len(elements)))
        properties["tensile_strength"] = sum(ratios) * 15 + (5 if "carbon" in elements else 0) # Carbon backbone for strength
        if "fluorine" in elements: properties["stability"] = 0.9 # PTFE-like
        score += properties["tensile_strength"] * 0.2 + properties["biodegradability"] * 0.3
    elif "alloy" in material_type:
        properties["conductivity"] = sum(ratios) * 60 + (20 if "copper" in elements else 0)
        properties["tensile_strength"] = sum(ratios) * 20 + (10 if "iron" in elements else 0)
        if "aluminum" in elements and "magnesium" in elements: properties["stability"] = 0.8
        score += properties["conductivity"] * 0.2 + properties["tensile_strength"] * 0.15

    # Add some randomness to make it less deterministic and simulate real-world variability
    import random
    score += random.uniform(-10, 10)
    score = max(0, min(100, score)) # Cap score between 0 and 100

    # Round properties for cleaner output
    for k in properties: properties[k] = round(properties[k], 2)

    print(f"Predicted properties: {properties}, Score: {score:.2f}")
    return {"predicted_properties": properties, "score": score, "composition": material_composition}

# List of all tools available to agents
tools = [material_properties_predictor]
tool_executor = ToolExecutor(tools)

# --- 3. Initialize LLM ---
# Using a powerful LLM for agent reasoning, 2026 models are highly capable
llm = ChatOpenAI(model="gpt-4o-2024-05-13", temperature=0.5)

# --- 4. Define Agent Prompts ---
# Base prompt for agents that can use tools, leveraging `create_tool_calling_agent`
# This prompt structure is optimized for tool-calling LLMs.

# Supervisor prompt
supervisor_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a research director overseeing a team of material science agents."
     "Your goal is to guide the team to discover a novel material with specific properties."
     "Based on the current state of research, material proposals, and evaluation results,"
     "you must decide the next step. Available steps are: 'research', 'hypothesis', 'evaluate', 'refine', 'report', 'end'."
     "You have a maximum of {max_iterations} iterations. Current iteration: {iterations}."
     "If a material scores 70 or higher, it is considered satisfactory."
     "Respond with only the name of the next node to transition to."
     "Current Request: {initial_request}"
     "Research Findings: {research_findings}"
     "Last Proposal: {current_proposal}"
     "Last Evaluation: {last_evaluation_score}"
     "Messages: {messages}"),
    MessagesPlaceholder(variable_name="messages")
])

# Agent prompts for tool-calling agents
def create_agent(llm: ChatOpenAI, tools: list, system_prompt: str):
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt + "\n\nYou have access to the following tools: {tools}"),
        MessagesPlaceholder(variable_name="messages"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ])
    return create_tool_calling_agent(llm, tools, prompt)

research_agent_system_prompt = (
    "You are a Material Science Researcher. Your task is to gather information about existing materials "
    "and scientific principles relevant to the initial request. Focus on identifying key elements, "
    "structures, or synthesis methods. Summarize your findings concisely. Do not propose materials yet."
)
hypothesis_agent_system_prompt = (
    "You are a Material Design Engineer. Based on the research findings, your task is to propose "
    "a novel material composition. The proposal must be a dictionary with 'type' (e.g., 'polymer', 'alloy'), "
    "'elements' (list of strings), 'ratios' (list of floats, summing to 1 or close), and 'notes' (string). "
    "Use the `material_properties_predictor` tool to format your output, but do not call it. "
    "Only output the tool call for `material_properties_predictor` with your proposed material as arguments."
)
refine_agent_system_prompt = (
    "You are a Material Optimization Specialist. Your task is to refine the previous material proposal "
    "based on its evaluation results. Analyze the predicted properties and score to make intelligent adjustments "
    "to the composition (elements, ratios) or type. Propose a new material using the `material_properties_predictor` tool "
    "format. Only output the tool call for `material_properties_predictor` with your refined material as arguments."
)
report_agent_system_prompt = (
    "You are a Scientific Reporter. Your task is to compile a comprehensive report summarizing the entire R&D process. "
    "Include the initial request, key research findings, all proposed materials with their evaluations, "
    "and a final recommendation (if a satisfactory material was found). Present the information clearly and professionally."
)

# --- 5. Define Agent Nodes ---
# These are the actual agent functions that will be called by the graph.
# They update the state and return it.

class AgentNode:
    def __init__(self, agent_executor: Any, name: str):
        self.agent_executor = agent_executor
        self.name = name

    def __call__(self, state: ResearchState) -> ResearchState:
        print(f"\n--- {self.name} Activated ---")
        # The agent receives the full state, but we only pass relevant messages to the LLM
        result = self.agent_executor.invoke({"messages": state["messages"]})
        # Update messages with the agent's response
        state["messages"].append(AIMessage(content=result["output"]))
        return {"messages": state["messages"]}

# For tool-using agents, we need a slightly different structure to handle tool calls
class ToolAgentNode:
    def __init__(self, agent_executor: Any, tool_executor: ToolExecutor, name: str):
        self.agent_executor = agent_executor
        self.tool_executor = tool_executor
        self.name = name

    def __call__(self, state: ResearchState) -> ResearchState:
        print(f"\n--- {self.name} Activated ---")
        agent_outcome = self.agent_executor.invoke({"messages": state["messages"]})
        state["messages"].append(AIMessage(content=agent_outcome.return_values["output"]))

        # If the agent wants to call a tool, execute it
        if agent_outcome.tool_calls:
            for tool_call in agent_outcome.tool_calls:
                print(f"Executing tool: {tool_call.name} with args {tool_call.args}")
                tool_output = self.tool_executor.invoke(tool_call)
                state["messages"].append(ToolMessage(content=str(tool_output), tool_call_id=tool_call.id))
                # Specific logic for material_properties_predictor tool
                if tool_call.name == "material_properties_predictor":
                    # Update state with the new proposal and evaluation result
                    proposals = state.get("material_proposals", [])
                    proposals.append(tool_output["composition"])
                    eval_results = state.get("evaluation_results", [])
                    eval_results.append(tool_output)
                    state["current_proposal"] = tool_output["composition"]
                    state["material_proposals"] = proposals
                    state["evaluation_results"] = eval_results
        return state

# Create the actual agents
research_agent_executor = create_agent(llm, [], research_agent_system_prompt)
hypothesis_agent_executor = create_agent(llm, tools, hypothesis_agent_system_prompt)
refine_agent_executor = create_agent(llm, tools, refine_agent_system_prompt)
report_agent_executor = create_agent(llm, [], report_agent_system_prompt)

# Wrap them in our custom nodes
research_node = AgentNode(research_agent_executor, "Research Agent")
hypothesis_node = ToolAgentNode(hypothesis_agent_executor, tool_executor, "Hypothesis Agent")
refine_node = ToolAgentNode(refine_agent_executor, tool_executor, "Refinement Agent")
report_node = AgentNode(report_agent_executor, "Reporting Agent")

# The evaluate step is essentially just calling the tool, so we can use a ToolNode directly
evaluate_node = ToolNode(tools)

# Supervisor logic as a function
def call_supervisor(state: ResearchState) -> ResearchState:
    print("\n--- Supervisor Activated ---")
    current_messages = state["messages"] + [
        HumanMessage(content=f"Current Request: {state['initial_request']}"),
        HumanMessage(content=f"Research Findings: {state['research_findings']}"),
        HumanMessage(content=f"Last Proposal: {state['current_proposal']}"),
        HumanMessage(content=f"Last Evaluation Score: {state['evaluation_results'][-1]['score'] if state['evaluation_results'] else 'N/A'}")
    ]

    # Use LLM to decide next step
    supervisor_response = llm.invoke(supervisor_prompt.format_messages(
        max_iterations=state["max_iterations"],
        iterations=state["iterations"],
        initial_request=state["initial_request"],
        research_findings=state["research_findings"],
        current_proposal=state["current_proposal"],
        last_evaluation_score=state["evaluation_results"][-1]["score"] if state["evaluation_results"] else 'N/A',
        messages=current_messages
    ))
    next_node = supervisor_response.content.strip().lower()
    print(f"Supervisor decided next node: {next_node}")

    # Increment iteration count if not ending or reporting
    if next_node not in ["end", "report"]:
        state["iterations"] += 1

    return {"next_node": next_node, "messages": state["messages"] + [AIMessage(content=f"Supervisor decision: {next_node}")]}

# --- 6. Build the Graph ---
workflow = StateGraph(ResearchState)

# Add nodes for each agent/step
workflow.add_node("research", research_node)
workflow.add_node("hypothesis", hypothesis_node)
workflow.add_node("evaluate", evaluate_node)
workflow.add_node("refine", refine_node)
workflow.add_node("report", report_node)
workflow.add_node("supervisor", call_supervisor)

# Set the entry point to the supervisor
workflow.set_entry_point("supervisor")

# Add conditional edges from the supervisor
# The supervisor's output (`next_node` in state) determines the next step
workflow.add_conditional_edges(
    "supervisor",
    lambda state: state["next_node"],
    {
        "research": "research",
        "hypothesis": "hypothesis",
        "evaluate": "evaluate",
        "refine": "refine",
        "report": "report",
        "end": END
    }
)

# Add edges from agents back to the supervisor for decision making
# After each agent completes its task, it returns control to the supervisor
workflow.add_edge("research", "supervisor")
workflow.add_edge("hypothesis", "supervisor")
workflow.add_edge("evaluate", "supervisor")
workflow.add_edge("refine", "supervisor")

# The report node is a terminal node, it finishes the process
workflow.add_edge("report", END)

# Compile the graph
app = workflow.compile()

# --- 7. Example Invocation ---
# Initial state for a new R&D request
initial_state = {
    "initial_request": "a biodegradable polymer with high tensile strength for packaging",
    "research_findings": [],
    "material_proposals": [],
    "current_proposal": None,
    "evaluation_results": [],
    "iterations": 0,
    "max_iterations": 5, # Limit iterations to prevent infinite loops in assessment
    "messages": [HumanMessage(content="Starting R&D for new material.")],
    "next_node": "supervisor" # Initial decision point
}

print("\n--- Starting R&D Assistant Workflow ---")
for s in app.stream(initial_state):
    # LangGraph streams state changes. Print the last updated node.
    if "__end__" not in s:
        print(list(s.keys())[0], s[list(s.keys())[0]])

print("\n--- Workflow Completed ---")
final_state = app.get_state(initial_state).values

print("\nFinal State Summary:")
print(f"Initial Request: {final_state['initial_request']}")
print(f"Total Iterations: {final_state['iterations']}")
print(f"Research Findings: {final_state['research_findings']}")
print(f"All Proposals ({len(final_state['material_proposals'])}):")
for i, prop in enumerate(final_state['material_proposals']):
    eval_res = final_state['evaluation_results'][i] if i < len(final_state['evaluation_results']) else {"score": "N/A"}
    print(f"  - Proposal {i+1}: {prop} (Score: {eval_res['score']:.2f})")

successful_material = next((res for res in final_state['evaluation_results'] if res.get('score', 0) >= 70), None)
if successful_material:
    print(f"\nSUCCESS! Recommended Material: {successful_material['composition']} (Score: {successful_material['score']:.2f})")
else:
    print("\nNo material met the criteria within the given iterations.")

# --- Time-Travel Debugging Example (Conceptual) ---
# To demonstrate time-travel debugging, you would typically use LangSmith (LangChain's observability platform).
# Each step in the `app.stream` above corresponds to a state transition.
# In LangSmith, you can view the state at each node, the inputs/outputs of agents/tools,
# and trace the exact path taken by the graph. If a bug occurred (e.g., an agent generated
# an invalid proposal), you could pinpoint the exact iteration and agent responsible,
# inspect its inputs (messages, current state), and understand why it made that decision.
# For example, if 'refine' agent consistently lowers the score, you could inspect its prompt
# and the feedback it received from 'evaluate' at each step to identify the flaw.
print("\n--- Time-Travel Debugging (Conceptual) ---")
print("To perform time-travel debugging, observe the LangSmith trace generated for this run.")
print("You can inspect the state at each node, agent inputs/outputs, and tool calls to understand the flow and diagnose issues.")
print("For instance, if a material proposal consistently fails, you can trace back to the 'hypothesis' or 'refine' agent's inputs and prompt to see why it's making suboptimal choices.")
